## Timeseries for CONUS and Europe

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# timeseries_total_events_obs_vs_xgbglobal_regions.py
#
# Creates a 2-panel yearly time series figure:
#   - top: CONUS
#   - bottom: Europe
#
# For each region:
#   - observations on left y-axis
#   - xgbGlobal predictions on right y-axis
#   - linear trend line for predictions
#
# Metric plotted:
#   total number of significant-hail gridcell-events per year
#   over the whole regional domain
#
# Periods:
#   CONUS observations: 1959–2024 excluding missing years
#   CONUS predictions : 1959–2024 excluding missing years
#
#   Europe observations: 1979–2022 excluding missing years
#   Europe predictions : 1959–2024 excluding missing years
#
# Prediction input:
#   uses already saved yearly regional prediction files:
#     pred_US_YYYY_fold5.npz
#     pred_EU_YYYY_fold5.npz
#
# Observation input:
#   NOAA / ESSL gridded hail files
#
# Notes:
#   - observations are converted to significant hail using threshold_cm
#   - predictions are read from mean_annual and converted back to
#     total modeled hail gridcell-events using ndays in the year
#   - all curves have lw=1.0
#   - predicted curves use color #d81b60
#   - light grey background is shown from 2000–2022
# ============================================================

from pathlib import Path
from datetime import datetime
import calendar
import numpy as np
import pandas as pd
from netCDF4 import Dataset
import matplotlib.pyplot as plt


# ============================================================
# 1) CONFIG
# ============================================================
base_cumulus = Path("/nfs/cumulus/highres_nobackup/agebhardt")
base_cluster = Path("/cluster/home/agebhardt")

cfg = {
    "fold": 0,
    "threshold_cm": 4.4,
    "missing_years": [1966, 1979, 1999, 2003],
    "pred_start": 1959,
    "pred_end": 2024,
    "eu_obs_start": 1979,
    "eu_obs_end": 2022,
    "dpi": 200,
    "figsize": (14, 9),
    "pred_color": "#d81b60",
    "lw": 1.0,
    "shade_start": 2000,
    "shade_end": 2022,
}

paths = {
    "noaa_obs": base_cumulus / "hail_observations" / "SPC_data_griddedERA",
    "essl_obs": base_cumulus / "hail_observations" / "ESSL_data_griddedERA",
    "pred_us": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_US" / f"fold{cfg['fold']}_dall",
    "pred_eu": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_EU" / f"fold{cfg['fold']}_dall",
    "out": base_cluster / "plots" / "preds_1959-2024_xgbGlobal",
}
paths["out"].mkdir(parents=True, exist_ok=True)

outpath = paths["out"] / (
    f"timeseries_total_events_obs_vs_xgbglobal_conus_europe_fold{cfg['fold']}_"
    f"{cfg['pred_start']}-{cfg['pred_end']}.png"
)

# regional boxes used for cropping observations
us_lon_min, us_lon_max = -127.0, -66.0
us_lat_min, us_lat_max = 24.0, 50.0

eu_lon_min, eu_lon_max = -10.0, 35.0
eu_lat_min, eu_lat_max = 35.0, 65.0


# ============================================================
# 2) HELPERS
# ============================================================
def filtered_daily_time(start_year, end_year, missing_years):
    dates = pd.date_range(
        datetime(start_year, 1, 1, 0),
        end=datetime(end_year, 12, 31, 23),
        freq="D",
    )
    dates = dates[~dates.year.isin(missing_years)]
    return dates


def filtered_years(start_year, end_year, missing_years):
    years = np.arange(start_year, end_year + 1, dtype=int)
    return years[~np.isin(years, missing_years)]


def build_sig_hail_obs(obs_2var, threshold_cm):
    """
    obs_2var shape: (2, T, H, W)
      obs_2var[0] = hail occurrence
      obs_2var[1] = hail size
    returns:
      sig_hail: (T, H, W)
    """
    hail_bin = obs_2var[0].astype(np.float32)
    hail_size = obs_2var[1].astype(np.float32)

    sig = (hail_bin == 1) & np.isfinite(hail_size) & (hail_size >= threshold_cm)
    return sig.astype(np.float32)


def yearly_region_total_hail_events(hail_3d, dates):
    """
    hail_3d: (T, H, W), binary daily hail occurrence per grid cell
    returns:
      years_unique, vals
    where vals = total annual hail gridcell-events over the region
    """
    years_unique = np.unique(dates.year)
    vals = np.full(years_unique.shape, np.nan, dtype=np.float32)

    for i, y in enumerate(years_unique):
        mask = dates.year == y
        hail_y = hail_3d[mask]  # (days, H, W)
        vals[i] = np.nansum(hail_y)

    return years_unique.astype(int), vals


def load_regional_prediction_timeseries_total_events(pred_dir, pred_prefix, years, fold):
    """
    Reads saved regional yearly prediction files and returns:
    total modeled hail gridcell-events per year over the whole region
    """
    vals = []
    yrs = []

    for year in years:
        path = pred_dir / f"{pred_prefix}_{int(year)}_fold{fold}.npz"
        if not path.is_file():
            print(f"[pred {pred_prefix} {year}] missing -> skip")
            continue

        data = np.load(path)
        if "mean_annual" not in data:
            raise RuntimeError(f"'mean_annual' missing in {path}")

        mean_annual = data["mean_annual"].astype(np.float32)  # fraction of days per year
        ndays = 365 + int(calendar.isleap(int(year)))

        # total modeled hail gridcell-events in the region for this year
        vals.append(np.nansum(mean_annual * ndays))
        yrs.append(int(year))

    return np.array(yrs, dtype=int), np.array(vals, dtype=np.float32)


def fit_linear_trend(years, values):
    mask = np.isfinite(values)
    if np.sum(mask) < 2:
        return np.full(values.shape, np.nan), np.nan

    coef = np.polyfit(years[mask], values[mask], 1)
    slope_per_year = coef[0]
    trend = np.polyval(coef, years)

    mean_val = np.nanmean(values[mask])
    slope_pct_per_decade = (slope_per_year / mean_val) * 100 * 10

    return trend, slope_pct_per_decade


def read_noaa_observations_whole_period(noaa_dir, years, lon_min, lon_max, lat_min, lat_max):
    hail_vars = ["Hail", "HailSize"]

    idx_lat = None
    idx_lon = None
    rgrLat = None
    rgrLon = None
    obs = None

    total_days = sum(365 + int(calendar.isleap(int(y))) for y in years)
    dd = 0

    for y in years:
        nc_path = noaa_dir / f"SPC-Hail-StormReports_gridded-75km_{int(y)}.nc"
        if not nc_path.is_file():
            raise FileNotFoundError(f"Missing NOAA file: {nc_path}")

        with Dataset(str(nc_path), mode="r") as nc:
            lat1d_full = np.squeeze(nc.variables["lat"][:, 0]).astype(np.float32)
            lon1d_full = np.squeeze(nc.variables["lon"][0, :]).astype(np.float32)

            if idx_lat is None:
                idx_lat = np.where((lat1d_full >= lat_min) & (lat1d_full <= lat_max))[0]
                idx_lon = np.where((lon1d_full >= lon_min) & (lon1d_full <= lon_max))[0]

                if idx_lat.size == 0 or idx_lon.size == 0:
                    raise RuntimeError("NOAA bbox produced empty subset.")

                ny = idx_lat.size
                nx = idx_lon.size
                obs = np.zeros((2, total_days, ny, nx), dtype=np.float32)

                lat1d = lat1d_full[idx_lat]
                lon1d = lon1d_full[idx_lon]
                rgrLon = np.tile(lon1d[None, :], (lat1d.size, 1))
                rgrLat = np.tile(lat1d[:, None], (1, lon1d.size))

                print("NOAA cropped lat range:", float(lat1d.min()), float(lat1d.max()))
                print("NOAA cropped lon range:", float(lon1d.min()), float(lon1d.max()))
                print("NOAA cropped shape:", ny, nx)

            yearlength = 365 + int(calendar.isleap(int(y)))

            for ii, vname in enumerate(hail_vars):
                data_full = np.squeeze(nc.variables[vname][:]).astype(np.float32)
                obs[ii, dd:dd + yearlength, :, :] = data_full[:, idx_lat, :][:, :, idx_lon]

        dd += yearlength

    return obs, rgrLat, rgrLon


def read_essl_observations_whole_period(essl_dir, years, lon_min, lon_max, lat_min, lat_max):
    size_name_candidates = ["HailSize", "size", "Size"]

    idx_lat = None
    idx_lon = None
    rgrLat = None
    rgrLon = None
    obs = None
    size_varname = None

    total_days = sum(365 + int(calendar.isleap(int(y))) for y in years)
    dd = 0

    for y in years:
        nc_path = essl_dir / f"ESSL-Hail-StormReports_gridded-ERA5_{int(y)}.nc"
        if not nc_path.is_file():
            raise FileNotFoundError(f"Missing ESSL file: {nc_path}")

        with Dataset(str(nc_path), mode="r") as nc:
            if idx_lat is None:
                lat1d_full = np.squeeze(nc.variables["lat"][:, 0]).astype(np.float32)
                lon1d_full = np.squeeze(nc.variables["lon"][0, :]).astype(np.float32)
                lon1d_full = np.where(lon1d_full > 180, lon1d_full - 360, lon1d_full)

                idx_lat = np.where((lat1d_full >= lat_min) & (lat1d_full <= lat_max))[0]
                idx_lon = np.where((lon1d_full >= lon_min) & (lon1d_full <= lon_max))[0]

                if idx_lat.size == 0 or idx_lon.size == 0:
                    raise RuntimeError("ESSL bbox produced empty subset.")

                for cand in size_name_candidates:
                    if cand in nc.variables:
                        size_varname = cand
                        break

                if size_varname is None:
                    raise KeyError(
                        f"No hail size variable found in {nc_path.name}. "
                        f"Tried: {size_name_candidates}. Available: {list(nc.variables.keys())}"
                    )

                ny = idx_lat.size
                nx = idx_lon.size
                obs = np.zeros((2, total_days, ny, nx), dtype=np.float32)

                lat1d = lat1d_full[idx_lat]
                lon1d = lon1d_full[idx_lon]
                rgrLon = np.tile(lon1d[None, :], (lat1d.size, 1))
                rgrLat = np.tile(lat1d[:, None], (1, lon1d.size))

                print("ESSL size variable detected:", size_varname)
                print("ESSL cropped lat range:", float(lat1d.min()), float(lat1d.max()))
                print("ESSL cropped lon range:", float(lon1d.min()), float(lon1d.max()))
                print("ESSL cropped shape:", ny, nx)

            yearlength = 365 + int(calendar.isleap(int(y)))

            hail_full = np.squeeze(nc.variables["Hail"][:]).astype(np.float32)
            hail_crop = hail_full[:, idx_lat, :][:, :, idx_lon]

            size_full = np.squeeze(nc.variables[size_varname][:]).astype(np.float32)
            size_crop = size_full[:, idx_lat, :][:, :, idx_lon]

            obs[0, dd:dd + yearlength, :, :] = hail_crop
            obs[1, dd:dd + yearlength, :, :] = size_crop

        dd += yearlength

    return obs, rgrLat, rgrLon


def plot_timeseries(
    years_obs_us,
    obs_us,
    years_pred_us,
    pred_us,
    years_obs_eu,
    obs_eu,
    years_pred_eu,
    pred_eu,
    outpath,
):
    fig, axes = plt.subplots(2, 1, figsize=cfg["figsize"], dpi=cfg["dpi"], sharex=True)

    pred_color = cfg["pred_color"]
    lw = cfg["lw"]

    # ---------------- CONUS ----------------
    ax = axes[0]
    axr = ax.twinx()

    ax.axvspan(2000, 2008, color="0.9", zorder=0)
    ax.axvspan(2008, 2022, color="0.82", zorder=0)

    line_obs_us, = ax.plot(
        years_obs_us, obs_us,
        color="black", lw=lw, label="Observations"
    )
    line_pred_us, = axr.plot(
        years_pred_us, pred_us,
        color=pred_color, lw=lw, label="Predictions"
    )

    trend_us, slope_pct_us = fit_linear_trend(years_pred_us, pred_us)
    line_trend_us, = axr.plot(
        years_pred_us,
        trend_us,
        color=pred_color,
        lw=lw,
        linestyle="--",
        label=f"Trend: {slope_pct_us:.1f}% / decade"
    )

    ax.set_title("CONUS", fontsize=12)
    # ax.set_ylabel("Observed significant hail events per year", color="black",fontsize=14)
    # axr.set_ylabel("Modeled significant hail events per year", color=pred_color,fontsize=14)

    ax.tick_params(axis="y", colors="black",labelsize=14)
    axr.tick_params(axis="y", colors=pred_color,labelsize=14)
    ax.grid(True, alpha=0.25)

    ax.legend(
        [line_obs_us, line_pred_us, line_trend_us],
        ["Observations", "Predictions", f"Trend: {slope_pct_us:.1f}% / decade"],
        loc="upper left",
        fontsize=14,
        frameon=False,
    )

    # ---------------- Europe ----------------
    ax = axes[1]
    axr = ax.twinx()

    ax.axvspan(2000, 2008, color="0.9", zorder=0)
    ax.axvspan(2008, 2022, color="0.82", zorder=0)

    line_obs_eu, = ax.plot(
        years_obs_eu, obs_eu,
        color="black", lw=lw, label="Observations"
    )
    line_pred_eu, = axr.plot(
        years_pred_eu, pred_eu,
        color=pred_color, lw=lw, label="Predictions"
    )

    trend_eu, slope_pct_eu = fit_linear_trend(years_pred_eu, pred_eu)
    line_trend_eu, = axr.plot(
        years_pred_eu,
        trend_eu,
        color=pred_color,
        lw=lw,
        linestyle="--",
        label=f"Trend: {slope_pct_eu:.1f}% / decade",
    )

    ax.set_title("Europe", fontsize=14)
    # ax.set_ylabel("Observed significant hail events per year", color="black", fontsize=14)
    # axr.set_ylabel("Modeled significant hail events per year", color=pred_color, fontsize=14)
    ax.set_xlabel("Year", fontsize=14)

    ax.tick_params(axis="y", colors="black",labelsize=14)
    axr.tick_params(axis="y", colors=pred_color, labelsize=14)
    ax.grid(True, alpha=0.25)

    ax.legend(
        [line_obs_eu, line_pred_eu, line_trend_eu],
        ["Observations", "Predictions", f"Trend: {slope_pct_eu:.1f}% / decade"],
        loc="upper left",
        fontsize=14,
        frameon=False,
    )

    axes[1].set_xlim(cfg["pred_start"], cfg["pred_end"])
    axes[1].set_xticks(np.arange(cfg["pred_start"], cfg["pred_end"] + 1, 5))
    axes[1].tick_params(axis="x", labelsize=14)

    fig.suptitle(
        f"Observed and Modeled Significant Hail Events per Year\n"
        f"xgbGlobal (Fold {cfg['fold']}) | CONUS and Europe",
        fontsize=14,
        y=0.98,
    )

    fig.text(
        0.015, 0.5,
        "Observed significant hail events per year",
        va="center",
        rotation="vertical",
        fontsize=14,
        color="black",
    )
    
    fig.text(
        0.985, 0.5,
        "Modeled significant hail events per year",
        va="center",
        rotation="vertical",
        fontsize=14,
        color=pred_color,
    )

    fig.tight_layout(rect=[0.05, 0, 0.95, 0.96])
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 3) TIME AXES
# ============================================================
years_pred = filtered_years(cfg["pred_start"], cfg["pred_end"], cfg["missing_years"])
dates_pred = filtered_daily_time(cfg["pred_start"], cfg["pred_end"], cfg["missing_years"])

years_eu_obs = filtered_years(cfg["eu_obs_start"], cfg["eu_obs_end"], cfg["missing_years"])
dates_eu_obs = filtered_daily_time(cfg["eu_obs_start"], cfg["eu_obs_end"], cfg["missing_years"])

print("Prediction years:", years_pred[0], "to", years_pred[-1], "| n =", len(years_pred))
print("Europe obs years:", years_eu_obs[0], "to", years_eu_obs[-1], "| n =", len(years_eu_obs))


# ============================================================
# 4) LOAD OBSERVATIONS
# ============================================================
print("\nLoading NOAA observations ...")
rgrNOAAObs, rgrLatNOAA, rgrLonNOAA = read_noaa_observations_whole_period(
    noaa_dir=paths["noaa_obs"],
    years=years_pred,
    lon_min=us_lon_min,
    lon_max=us_lon_max,
    lat_min=us_lat_min,
    lat_max=us_lat_max,
)
print("rgrNOAAObs shape:", rgrNOAAObs.shape)
print("len(dates_pred):", len(dates_pred))

print("\nLoading ESSL observations ...")
rgrESSLObs, rgrLatESSL, rgrLonESSL = read_essl_observations_whole_period(
    essl_dir=paths["essl_obs"],
    years=years_eu_obs,
    lon_min=eu_lon_min,
    lon_max=eu_lon_max,
    lat_min=eu_lat_min,
    lat_max=eu_lat_max,
)
print("rgrESSLObs shape:", rgrESSLObs.shape)
print("len(dates_eu_obs):", len(dates_eu_obs))


# ============================================================
# 5) COMPUTE OBSERVATION TIME SERIES
# ============================================================
print("\nComputing observation time series ...")

sig_hail_us = build_sig_hail_obs(rgrNOAAObs, threshold_cm=cfg["threshold_cm"])
years_obs_us, obs_us = yearly_region_total_hail_events(sig_hail_us, dates_pred)

sig_hail_eu = build_sig_hail_obs(rgrESSLObs, threshold_cm=cfg["threshold_cm"])
years_obs_eu, obs_eu = yearly_region_total_hail_events(sig_hail_eu, dates_eu_obs)

print("CONUS obs years:", years_obs_us[0], years_obs_us[-1], "| n =", len(years_obs_us))
print("Europe obs years:", years_obs_eu[0], years_obs_eu[-1], "| n =", len(years_obs_eu))


# ============================================================
# 6) LOAD PREDICTION TIME SERIES
# ============================================================
print("\nLoading saved regional predictions ...")

years_pred_us, pred_us = load_regional_prediction_timeseries_total_events(
    pred_dir=paths["pred_us"],
    pred_prefix="pred_US",
    years=years_pred,
    fold=cfg["fold"],
)

years_pred_eu, pred_eu = load_regional_prediction_timeseries_total_events(
    pred_dir=paths["pred_eu"],
    pred_prefix="pred_EU",
    years=years_pred,
    fold=cfg["fold"],
)

print("CONUS pred years:", years_pred_us[0], years_pred_us[-1], "| n =", len(years_pred_us))
print("Europe pred years:", years_pred_eu[0], years_pred_eu[-1], "| n =", len(years_pred_eu))


# ============================================================
# 7) SUMMARY
# ============================================================
print("\nSummary")
print("-------")
print(f"CONUS observations min/max: {np.nanmin(obs_us):.3f} / {np.nanmax(obs_us):.3f}")
print(f"CONUS predictions  min/max: {np.nanmin(pred_us):.3f} / {np.nanmax(pred_us):.3f}")
print(f"Europe observations min/max: {np.nanmin(obs_eu):.3f} / {np.nanmax(obs_eu):.3f}")
print(f"Europe predictions  min/max: {np.nanmin(pred_eu):.3f} / {np.nanmax(pred_eu):.3f}")


# ============================================================
# 8) PLOT
# ============================================================
print("\nPlotting ...")
plot_timeseries(
    years_obs_us=years_obs_us,
    obs_us=obs_us,
    years_pred_us=years_pred_us,
    pred_us=pred_us,
    years_obs_eu=years_obs_eu,
    obs_eu=obs_eu,
    years_pred_eu=years_pred_eu,
    pred_eu=pred_eu,
    outpath=outpath,
)

print(f"\nSaved figure to:\n{outpath}")


# ============================================================
# CITY-LEVEL TIME SERIES: 3x3 GRID CELL AREA
# ============================================================

cities_us = {
    "Oklahoma City": {"lat": 35.4676, "lon": -97.5164},
    "Kansas City":  {"lat": 39.0997, "lon": -94.5786},
    "Dallas":       {"lat": 32.7767, "lon": -96.7970},
}

cities_eu = {
    "Graz":   {"lat": 47.0707, "lon": 15.4395},
    "Venice": {"lat": 45.4408, "lon": 12.3155},
    "Turin":  {"lat": 45.0703, "lon": 7.6869},
}


def nearest_grid_idx(lat2d, lon2d, lat0, lon0):
    dist2 = (lat2d - lat0) ** 2 + (lon2d - lon0) ** 2
    iy, ix = np.unravel_index(np.nanargmin(dist2), dist2.shape)
    return iy, ix


def get_3x3_slice(iy, ix, ny, nx):
    y0 = max(0, iy - 1)
    y1 = min(ny, iy + 2)
    x0 = max(0, ix - 1)
    x1 = min(nx, ix + 2)
    return slice(y0, y1), slice(x0, x1)


def yearly_city_total_obs(sig_hail, dates, lat2d, lon2d, city):
    iy, ix = nearest_grid_idx(lat2d, lon2d, city["lat"], city["lon"])
    ys, xs = get_3x3_slice(iy, ix, sig_hail.shape[1], sig_hail.shape[2])

    years = np.unique(dates.year)
    vals = []

    for y in years:
        mask = dates.year == y
        vals.append(np.nansum(sig_hail[mask, ys, xs]))

    return years.astype(int), np.array(vals, dtype=np.float32)


def yearly_city_total_pred(pred_dir, pred_prefix, years, fold, lat2d, lon2d, city):
    iy, ix = nearest_grid_idx(lat2d, lon2d, city["lat"], city["lon"])
    ys, xs = get_3x3_slice(iy, ix, lat2d.shape[0], lat2d.shape[1])

    vals = []
    yrs = []

    for year in years:
        path = pred_dir / f"{pred_prefix}_{int(year)}_fold{fold}.npz"
        if not path.is_file():
            print(f"[pred {pred_prefix} {year}] missing -> skip")
            continue

        data = np.load(path)
        mean_annual = data["mean_annual"].astype(np.float32)
        ndays = 365 + int(calendar.isleap(int(year)))

        vals.append(np.nansum(mean_annual[ys, xs] * ndays))
        yrs.append(int(year))

    return np.array(yrs, dtype=int), np.array(vals, dtype=np.float32)


def plot_city_timeseries(city_results, outpath):
    fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), dpi=cfg["dpi"], sharex=True)
    axes = axes.ravel()

    pred_color = cfg["pred_color"]

    for ax, result in zip(axes, city_results):
        city = result["city"]

        # twin axis
        axr = ax.twinx()

        # background shading
        ax.axvspan(2000, 2008, color="0.9", zorder=0)
        ax.axvspan(2008, 2022, color="0.82", zorder=0)

        # observations (LEFT axis)
        line_obs, = ax.plot(
            result["years_obs"],
            result["obs"],
            color="black",
            lw=1.0,
            label="Observations",
        )

        # predictions (RIGHT axis)
        line_pred, = axr.plot(
            result["years_pred"],
            result["pred"],
            color=pred_color,
            lw=1.0,
            label="Predictions",
        )

        # trend (RIGHT axis)
        trend, slope_pct = fit_linear_trend(result["years_pred"], result["pred"])
        line_trend, = axr.plot(
            result["years_pred"],
            trend,
            color=pred_color,
            lw=1.0,
            linestyle="--",
            label=f"Trend: {slope_pct:.1f}% / decade",
        )

        # styling
        ax.set_title(city, fontsize=13)
        ax.grid(True, alpha=0.25)

        ax.tick_params(axis="y", labelsize=11, colors="black")
        axr.tick_params(axis="y", labelsize=11, colors=pred_color)
        ax.tick_params(axis="x", labelsize=11)

        # combined legend
        ax.legend(
            [line_obs, line_pred, line_trend],
            ["Observations", "Predictions", f"Trend: {slope_pct:.1f}% / decade"],
            fontsize=8,
            frameon=False,
            loc="upper left",
        )

    # x-labels
    for ax in axes[3:]:
        ax.set_xlabel("Year", fontsize=12)

    # global y-labels
    fig.text(
        0.01, 0.5,
        "Observed significant hail events per year",
        va="center",
        rotation="vertical",
        fontsize=13,
        color="black",
    )

    fig.text(
        0.99, 0.5,
        "Modeled significant hail events per year",
        va="center",
        rotation="vertical",
        fontsize=13,
        color=pred_color,
        ha="right",
    )

    fig.suptitle(
        f"City-Level Significant Hail Time Series | xgbGlobal Fold {cfg['fold']}",
        fontsize=14,
        y=0.98,
    )

    fig.tight_layout(rect=[0.04, 0.02, 0.96, 0.95])
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# BUILD CITY TIME SERIES
# ============================================================

city_results = []

# CONUS cities
for name, city in cities_us.items():
    y_obs, obs_vals = yearly_city_total_obs(
        sig_hail=sig_hail_us,
        dates=dates_pred,
        lat2d=rgrLatNOAA,
        lon2d=rgrLonNOAA,
        city=city,
    )

    y_pred, pred_vals = yearly_city_total_pred(
        pred_dir=paths["pred_us"],
        pred_prefix="pred_US",
        years=years_pred,
        fold=cfg["fold"],
        lat2d=rgrLatNOAA,
        lon2d=rgrLonNOAA,
        city=city,
    )

    city_results.append({
        "city": name,
        "years_obs": y_obs,
        "obs": obs_vals,
        "years_pred": y_pred,
        "pred": pred_vals,
    })


# European cities
for name, city in cities_eu.items():
    y_obs, obs_vals = yearly_city_total_obs(
        sig_hail=sig_hail_eu,
        dates=dates_eu_obs,
        lat2d=rgrLatESSL,
        lon2d=rgrLonESSL,
        city=city,
    )

    y_pred, pred_vals = yearly_city_total_pred(
        pred_dir=paths["pred_eu"],
        pred_prefix="pred_EU",
        years=years_pred,
        fold=cfg["fold"],
        lat2d=rgrLatESSL,
        lon2d=rgrLonESSL,
        city=city,
    )

    city_results.append({
        "city": name,
        "years_obs": y_obs,
        "obs": obs_vals,
        "years_pred": y_pred,
        "pred": pred_vals,
    })


outpath_cities = paths["out"] / (
    f"timeseries_city_level_3x3_okc_kc_dallas_graz_venice_turin_"
    f"xgbGlobal_fold{cfg['fold']}_{cfg['pred_start']}-{cfg['pred_end']}.png"
)

plot_city_timeseries(city_results, outpath_cities)

print(f"Saved city-level time-series figure to:\n{outpath_cities}")

Prediction years: 1959 to 2024 | n = 62
Europe obs years: 1980 to 2022 | n = 41

Loading NOAA observations ...
NOAA cropped lat range: 24.0 50.0
NOAA cropped lon range: -127.0 -66.0
NOAA cropped shape: 105 245
rgrNOAAObs shape: (2, 22647, 105, 245)
len(dates_pred): 22647

Loading ESSL observations ...
ESSL size variable detected: size
ESSL cropped lat range: 35.0 65.0
ESSL cropped lon range: -10.0 35.0
ESSL cropped shape: 121 181
rgrESSLObs shape: (2, 14976, 121, 181)
len(dates_eu_obs): 14976

Computing observation time series ...
CONUS obs years: 1959 2024 | n = 62
Europe obs years: 1980 2022 | n = 41

Loading saved regional predictions ...
CONUS pred years: 1959 2024 | n = 62
Europe pred years: 1959 2024 | n = 62

Summary
-------
CONUS observations min/max: 241.000 / 2640.000
CONUS predictions  min/max: 720.000 / 5061.000
Europe observations min/max: 0.000 / 373.000
Europe predictions  min/max: 107.000 / 2251.000

Plotting ...

Saved figure to:
/cluster/home/agebhardt/plots/preds_195

/scratch/tmp.65333791.agebhardt/ipykernel_1133798/3198391864.py:186: RuntimeWarning: invalid value encountered in scalar divide
  slope_pct_per_decade = (slope_per_year / mean_val) * 100 * 10


Saved city-level time-series figure to:
/cluster/home/agebhardt/plots/preds_1959-2024_xgbGlobal/timeseries_city_level_3x3_okc_kc_dallas_graz_venice_turin_xgbGlobal_fold0_1959-2024.png


## Global Timeseries

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# timeseries_global_predictions_only.py
#
# Creates one yearly global time series figure using saved xgbGlobal
# yearly prediction files:
#   pred_global_YYYY_fold5.npz
#
# Metric plotted:
#   total modeled significant-hail gridcell-events per year
#   over the whole global predictor domain
#
# Period:
#   1959–2024 excluding missing years
#
# Notes:
#   - uses saved yearly mean_annual fields
#   - converts mean_annual back to total modeled hail gridcell-events
#     using number of days in each year
#   - predicted curve color = #d81b60
#   - all curves have lw=1.0
#   - light grey background from 2000–2022
#   - trend shown in % / decade
# ============================================================

from pathlib import Path
from datetime import datetime
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1) CONFIG
# ============================================================
base_cumulus = Path("/nfs/cumulus/highres_nobackup/agebhardt")
base_cluster = Path("/cluster/home/agebhardt")

cfg = {
    "fold": 5,
    "missing_years": [1966, 1979, 1999, 2003],
    "pred_start": 1959,
    "pred_end": 2024,
    "dpi": 200,
    "figsize": (14, 5),
    "pred_color": "#d81b60",
    "lw": 1.0,
    "shade_start": 2000,
    "shade_end": 2022,
}

paths = {
    "pred_global": base_cumulus / "preds_1959-2024_xgbGlobal" / "preds_global" / f"fold{cfg['fold']}",
    "out": base_cluster / "plots" / "preds_1959-2024_xgbGlobal",
}
paths["out"].mkdir(parents=True, exist_ok=True)

outpath = paths["out"] / (
    f"timeseries_global_predictions_only_fold{cfg['fold']}_"
    f"{cfg['pred_start']}-{cfg['pred_end']}.png"
)


# ============================================================
# 2) HELPERS
# ============================================================
def filtered_years(start_year, end_year, missing_years):
    years = np.arange(start_year, end_year + 1, dtype=int)
    return years[~np.isin(years, missing_years)]


def load_global_prediction_timeseries_total_events(pred_dir, years, fold):
    """
    Reads saved global yearly prediction files and returns:
    total modeled hail gridcell-events per year over the whole global domain
    """
    vals = []
    yrs = []

    for year in years:
        path = pred_dir / f"pred_global_{int(year)}_fold{fold}.npz"
        if not path.is_file():
            print(f"[pred global {year}] missing -> skip")
            continue

        data = np.load(path)
        if "mean_annual" not in data:
            raise RuntimeError(f"'mean_annual' missing in {path}")

        mean_annual = data["mean_annual"].astype(np.float32)  # fraction of days per year
        ndays = 365 + int(calendar.isleap(int(year)))

        # total modeled hail gridcell-events in the whole global domain
        vals.append(np.nansum(mean_annual * ndays))
        yrs.append(int(year))

    return np.array(yrs, dtype=int), np.array(vals, dtype=np.float32)


def fit_linear_trend(years, values):
    mask = np.isfinite(values)
    if np.sum(mask) < 2:
        return np.full(values.shape, np.nan), np.nan

    coef = np.polyfit(years[mask], values[mask], 1)
    slope_per_year = coef[0]
    trend = np.polyval(coef, years)

    mean_val = np.nanmean(values[mask])
    if mean_val == 0:
        slope_pct_per_decade = np.nan
    else:
        slope_pct_per_decade = (slope_per_year / mean_val) * 100 * 10

    return trend, slope_pct_per_decade


def plot_timeseries(years_pred, pred_vals, outpath):
    fig, ax = plt.subplots(1, 1, figsize=cfg["figsize"], dpi=cfg["dpi"])

    pred_color = cfg["pred_color"]
    lw = cfg["lw"]

    ax.axvspan(cfg["shade_start"], cfg["shade_end"], color="0.9", zorder=0)

    line_pred, = ax.plot(
        years_pred,
        pred_vals,
        color=pred_color,
        lw=lw,
        label="Predictions",
    )

    trend, slope_pct = fit_linear_trend(years_pred, pred_vals)
    line_trend, = ax.plot(
        years_pred,
        trend,
        color=pred_color,
        lw=lw,
        linestyle="--",
        label=f"Trend: {slope_pct:.1f}% / decade",
    )

    ax.set_title("Global modeled significant hail events", fontsize=12)
    ax.set_ylabel("Modeled significant hail events per year", color=pred_color)
    ax.set_xlabel("Year")

    ax.tick_params(axis="y", colors=pred_color)
    ax.grid(True, alpha=0.25)

    ax.legend(
        [line_pred, line_trend],
        ["Predictions", f"Trend: {slope_pct:.1f}% / decade"],
        loc="upper left",
        fontsize=9,
        frameon=False,
    )

    ax.set_xlim(cfg["pred_start"], cfg["pred_end"])
    ax.set_xticks(np.arange(cfg["pred_start"], cfg["pred_end"] + 1, 5))

    fig.suptitle(
        f"Global modeled significant hail events per year\n"
        f"xgbGlobal (Fold {cfg['fold']})",
        fontsize=14,
        y=0.97,
    )

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 3) TIME AXIS
# ============================================================
years_pred = filtered_years(cfg["pred_start"], cfg["pred_end"], cfg["missing_years"])
print("Prediction years:", years_pred[0], "to", years_pred[-1], "| n =", len(years_pred))


# ============================================================
# 4) LOAD PREDICTION TIME SERIES
# ============================================================
print("\nLoading saved global predictions ...")

years_pred_global, pred_global = load_global_prediction_timeseries_total_events(
    pred_dir=paths["pred_global"],
    years=years_pred,
    fold=cfg["fold"],
)

print("Global pred years:", years_pred_global[0], years_pred_global[-1], "| n =", len(years_pred_global))


# ============================================================
# 5) SUMMARY
# ============================================================
print("\nSummary")
print("-------")
print(f"Global predictions min/max: {np.nanmin(pred_global):.3f} / {np.nanmax(pred_global):.3f}")


# ============================================================
# 6) PLOT
# ============================================================
print("\nPlotting ...")
plot_timeseries(
    years_pred=years_pred_global,
    pred_vals=pred_global,
    outpath=outpath,
)

print(f"\nSaved figure to:\n{outpath}")

Prediction years: 1959 to 2024 | n = 62

Loading saved global predictions ...
[pred global 2024] missing -> skip
Global pred years: 1959 2023 | n = 61

Summary
-------
Global predictions min/max: 1019.000 / 4457.000

Plotting ...

Saved figure to:
/cluster/home/agebhardt/plots/preds_1959-2024_xgbGlobal/timeseries_global_predictions_only_fold5_1959-2024.png
